In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model,
    calibration_model,
    SimpleAutoSort
)


In [2]:
file_dict = {
    'mouse1': {
        1215: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251215_232216',
        1217: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251217_225054',
        1219: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1&oldV1_natima_251219_201746'
    },
    'mouse2': {
        1214: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1B_natima_251214_154409',
        1215: '/media/ubuntu/sda/mouse_test/raw_data/WLF_V1left&128ch2mouse_natima_251215_223556',
        1216: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1left_natima_251216_214224',
        1217: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251217_220244',
        1218: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1od_natima_251218_214009',
        1219: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251219_192148'
    }

}

In [7]:
channel_list_A = ['A-000', 'A-001', 'A-002',
       'A-003', 'A-004', 'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
       'A-010', 'A-011', 'A-012', 'A-013', 'A-014', 'A-015', 'A-016',
       'A-017', 'A-018', 'A-019', 'A-020', 'A-021', 'A-022', 'A-023',
       'A-024', 'A-025', 'A-026', 'A-027', 'A-028', 'A-029', 'A-030',
       'A-031', 'A-032', 'A-033', 'A-034', 'A-035', 'A-036', 'A-037',
       'A-038', 'A-039', 'A-040', 'A-041', 'A-042', 'A-043', 'A-044',
       'A-045', 'A-046', 'A-047', 'A-048', 'A-049', 'A-050', 'A-051',
       'A-052', 'A-053', 'A-054', 'A-055', 'A-056', 'A-057', 'A-058',
       'A-059', 'A-060', 'A-061', 'A-062', 'A-063', 'A-064', 'A-065',
       'A-066', 'A-067', 'A-068', 'A-069', 'A-070', 'A-071', 'A-072',
       'A-073', 'A-074', 'A-075', 'A-076', 'A-077', 'A-078', 'A-079',
       'A-080', 'A-081', 'A-082', 'A-083', 'A-084', 'A-085', 'A-086',
       'A-087', 'A-088', 'A-089', 'A-090', 'A-091', 'A-092', 'A-093',
       'A-094', 'A-095', 'A-096', 'A-097', 'A-098', 'A-099', 'A-100',
       'A-101', 'A-102', 'A-103', 'A-104', 'A-105', 'A-106', 'A-107',
       'A-108', 'A-109', 'A-110', 'A-111', 'A-112', 'A-113', 'A-114',
       'A-115', 'A-116', 'A-117', 'A-118', 'A-119', 'A-120', 'A-121',
       'A-122', 'A-123', 'A-124', 'A-125', 'A-126', 'A-127']

channel_list_B = ['B-000', 'B-001', 'B-002',
       'B-003', 'B-004', 'B-005', 'B-006', 'B-007', 'B-008', 'B-009',
       'B-010', 'B-011', 'B-012', 'B-013', 'B-014', 'B-015', 'B-016',
       'B-017', 'B-018', 'B-019', 'B-020', 'B-021', 'B-022', 'B-023',
       'B-024', 'B-025', 'B-026', 'B-027', 'B-028', 'B-029', 'B-030',
       'B-031', 'B-032', 'B-033', 'B-034', 'B-035', 'B-036', 'B-037',
       'B-038', 'B-039', 'B-040', 'B-041', 'B-042', 'B-043', 'B-044',
       'B-045', 'B-046', 'B-047', 'B-048', 'B-049', 'B-050', 'B-051',
       'B-052', 'B-053', 'B-054', 'B-055', 'B-056', 'B-057', 'B-058',
       'B-059', 'B-060', 'B-061', 'B-062', 'B-063', 'B-064', 'B-065',
       'B-066', 'B-067', 'B-068', 'B-069', 'B-070', 'B-071', 'B-072',
       'B-073', 'B-074', 'B-075', 'B-076', 'B-077', 'B-078', 'B-079',
       'B-080', 'B-081', 'B-082', 'B-083', 'B-084', 'B-085', 'B-086',
       'B-087', 'B-088', 'B-089', 'B-090', 'B-091', 'B-092', 'B-093',
       'B-094', 'B-095', 'B-096', 'B-097', 'B-098', 'B-099', 'B-100',
       'B-101', 'B-102', 'B-103', 'B-104', 'B-105', 'B-106', 'B-107',
       'B-108', 'B-109', 'B-110', 'B-111', 'B-112', 'B-113', 'B-114',
       'B-115', 'B-116', 'B-117', 'B-118', 'B-119', 'B-120', 'B-121',
       'B-122', 'B-123', 'B-124', 'B-125', 'B-126', 'B-127']

In [8]:
mouse_name = 'mouse2'
combined_output_base = f'/media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/{mouse_name}'

In [10]:
# Clique级别测试流程（使用1217数据，在1215训练的模型上验证）
import torch

train_date = 1214

for test_date in [1215, 1216, 1217, 1218, 1219]:
    probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
    cliques = build_shank_cliques(probe, shank_boundaries=[250, 750, 1250])

    # 对每个clique进行处理
    for clique in cliques:
        clique_id = clique.clique_id

        # if clique_id != 0:
        #     continue
        print(f"\n{'='*60}")
        print(f"Processing Clique {clique_id}")
        print(f"{'='*60}")
        
        # 1. 加载1215（训练）和1217（测试）的neuron_inf
        train_data_folder = f'{combined_output_base}/clique_{clique_id}/date_{train_date}'
        test_data_folder = f'{combined_output_base}/clique_{clique_id}/date_{test_date}'
        
        train_neuron_inf_path = f'{train_data_folder}/neuron_inf.pickle'
        test_neuron_inf_path = f'{test_data_folder}/neuron_inf.pickle'
        
        if not os.path.exists(train_neuron_inf_path) or not os.path.exists(test_neuron_inf_path):
            print(f"  警告: 缺少neuron_inf文件，跳过clique {clique_id}")
            continue
        
        # 加载neuron_inf
        with open(train_neuron_inf_path, 'rb') as f:
            train_neuron_inf_dict = pickle.load(f)
        with open(test_neuron_inf_path, 'rb') as f:
            test_neuron_inf_dict = pickle.load(f)
        
        train_neuron_inf = neuron_inf_dict_to_dataframe(train_neuron_inf_dict)
        test_neuron_inf = neuron_inf_dict_to_dataframe(test_neuron_inf_dict)
        
        print(f"\n训练日期 {train_date}: {len(train_neuron_inf)} 个神经元")
        print(f"测试日期 {test_date}: {len(test_neuron_inf)} 个神经元")
        
        # 2. 比较1215和1217的neuron，找出重合、消失、新出现的（直接比较neuron ID）
        print(f"\n{'='*60}")
        print("比较神经元（直接比较neuron ID）")
        print(f"{'='*60}")
        
        train_neuron_ids = set(train_neuron_inf['Neuron'].unique())
        test_neuron_ids = set(test_neuron_inf['Neuron'].unique())
        
        # 统计结果
        matched_neuron_ids = train_neuron_ids & test_neuron_ids  # 重合的神经元
        disappeared_neuron_ids = train_neuron_ids - test_neuron_ids  # 消失的神经元
        new_neuron_ids = test_neuron_ids - train_neuron_ids  # 新出现的神经元
        
        print(f"\n统计结果:")
        print(f"  重合的神经元: {len(matched_neuron_ids)} (在{train_date}和{test_date}中都存在)")
        if len(matched_neuron_ids) > 0:
            print(f"    重合的神经元列表: {sorted(matched_neuron_ids)}")
        print(f"  消失的神经元: {len(disappeared_neuron_ids)} (在{train_date}中存在但在{test_date}中不存在)")
        if len(disappeared_neuron_ids) > 0:
            print(f"    消失的神经元列表: {sorted(disappeared_neuron_ids)}")
        print(f"  新出现的神经元: {len(new_neuron_ids)} (在{test_date}中存在但在{train_date}中不存在)")
        if len(new_neuron_ids) > 0:
            print(f"    新出现的神经元列表: {sorted(new_neuron_ids)}")
        
        # 为test_neuron_inf添加neuron_match列（用于calibration_model的评估）
        test_neuron_inf_matched = test_neuron_inf.copy()
        test_neuron_inf_matched['neuron_match'] = test_neuron_inf_matched['Neuron'].apply(
            lambda x: x if x in matched_neuron_ids else 'unmatch'
        )
        
        # 准备测试数据（只需要准备一次）
        data_path = file_dict[mouse_name][test_date]
        file_list_path = Path(data_path)
        rhd_files = list(file_list_path.glob("*.rhd"))
        file_list = sorted(rhd_files)
        
        if len(file_list) == 0:
            print(f"  警告: 在 {data_path} 中未找到.rhd文件，跳过")
            continue
        
        # 读取并合并该date的所有rhd文件
        recording_raw_list = []
        for file in file_list:
            recording_raw_list.append(se.read_intan(file, stream_id='0'))
        
        test_recording = concatenate_recordings(recording_list=recording_raw_list)
        
        # 检测通道类型并选择对应的channel_list
        available_channels = test_recording.get_channel_ids()
        if 'A-127' in available_channels:
            channel_list = channel_list_A
        elif 'B-127' in available_channels:
            channel_list = channel_list_B
        else:
            print(f"  警告: 未找到A-127或B-127通道，跳过")
            continue
        
        # 选择通道
        test_recording = test_recording.select_channels(channel_list)
        
        # 统一将B开头的channel重命名为A开头
        channel_ids = test_recording.get_channel_ids()
        new_channel_ids = []
        renamed_count = 0
        for ch_id in channel_ids:
            if isinstance(ch_id, str) and ch_id.startswith('B-'):
                new_ch_id = 'A-' + ch_id[2:]
                new_channel_ids.append(new_ch_id)
                renamed_count += 1
            else:
                new_channel_ids.append(ch_id)
        
        if renamed_count > 0:
            test_recording = test_recording.rename_channels(new_channel_ids)
        
        # 预处理recording
        test_recording = spre.unsigned_to_signed(test_recording)
        test_recording = spre.resample(test_recording, 10000)
        test_recording = spre.bandpass_filter(test_recording, freq_min=300, freq_max=3000)
        test_recording = spre.notch_filter(test_recording, freq=50)
        test_recording = spre.common_reference(test_recording, reference="global", operator="median")
        test_recording = test_recording.set_probegroup(probe)
        
        # 获取recording_clique
        test_recording_clique = get_recording_clique(test_recording, clique)
        print(f"  测试recording通道数: {len(test_recording_clique.get_channel_ids())}")
        
        clique_channel_ids = list(test_recording_clique.get_channel_ids())
        
        test_gt_detect_array_path = f'{test_data_folder}/gt_detect_array.csv'
        test_gt_detect_array = None
        if os.path.exists(test_gt_detect_array_path):
            test_gt_detect_array = pd.read_csv(test_gt_detect_array_path)
        
        # 重复实验5次，每次使用不同的模型权重
        n_repeats = 5
        n_channels = 32  # clique的通道数
        samplepoints = 30  # left_sample + right_sample = 10 + 20
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        for repeat_idx in range(1, n_repeats + 1):
            print(f"\n  ===== 重复实验 {repeat_idx}/{n_repeats} (使用 model_{repeat_idx}) =====")
            
            model_save_dir = f'{train_data_folder}/model_{repeat_idx}'
            noise_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_noise_clsfier.pth'
            label_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_label_clsfier.pth'
            
            # 检查模型文件是否存在
            if not os.path.exists(noise_model_path) or not os.path.exists(label_model_path):
                print(f"  警告: model_{repeat_idx} 的权重文件不存在，跳过")
                continue
            
            # 加载keep_id列表
            keep_id_path = f'{model_save_dir}/keep_id.pkl'
            if not os.path.exists(keep_id_path):
                print(f"  警告: keep_id.pkl不存在 {keep_id_path}，跳过")
                continue
            
            with open(keep_id_path, 'rb') as f:
                keep_id_list = pickle.load(f)
            
            # 创建模型（使用正确的构造函数参数）
            autosort_model = SimpleAutoSort(
                ch_num=n_channels,
                samplepoints=samplepoints,
                device=device,
                set_shank_id=keep_id_list,
                save_dir=model_save_dir,
                pos_weight_noise=None,  # 这些权重在推理时不需要
                pos_weight_label=None
            )
            
            autosort_model.clsfier_noise.load_state_dict(torch.load(noise_model_path, map_location=device))
            autosort_model.clsfier_label.load_state_dict(torch.load(label_model_path, map_location=device))
            autosort_model.eval()
            
            # 运行calibration
            calibration_results = calibration_model(
                recording_f=test_recording_clique,
                autosort_model=autosort_model,
                train_neuron_inf=train_neuron_inf,
                calibration_duration_seconds=300,
                n_additional_clusters=5,
                detection_params={
                    'thr_min': 2.5,
                    'thr_max': 10,
                    'distance': 6,
                    'wlen': 5,
                    'prominence': 10,
                    'max_firing_channel': 8,
                },
                window_params={
                    'left_sample': 10,
                    'right_sample': 20,
                },
                position_threshold=10.0,
                waveform_similarity_threshold=0.95,
                eval_neuron_inf=test_neuron_inf_matched,
                gt_detect_array=test_gt_detect_array,
                device=device
            )
            
            # 保存结果到不同的文件
            output_path = f"{test_data_folder}/calibration_model_{repeat_idx}.pkl"
            with open(output_path, 'wb') as f:
                pickle.dump(calibration_results, f)
            
            print(f"  重复实验 {repeat_idx}/{n_repeats} 完成，结果已保存到: {output_path}")
        # print(f"\n验证完成！")
        # print(f"  匹配的cluster数: {len(calibration_results['cluster_to_neuron_mapping'])}")
        # print(f"  匹配的神经元数: {len(calibration_results['neuron_to_clusters'])}")

    print("\n所有测试完成！")


[INFO] Shank 0: 32 channels (x range: min to 250)
[INFO] Shank 1: 32 channels (x range: 250 to 750)
[INFO] Shank 2: 32 channels (x range: 750 to 1250)
[INFO] Shank 3: 32 channels (x range: 1250 to max)
[INFO] Built 4 cliques from 4 shanks

Processing Clique 0

训练日期 1214: 17 个神经元
测试日期 1215: 16 个神经元

比较神经元（直接比较neuron ID）

统计结果:
  重合的神经元: 13 (在1214和1215中都存在)
    重合的神经元列表: [np.int64(1), np.int64(2), np.int64(4), np.int64(9), np.int64(10), np.int64(11), np.int64(17), np.int64(19), np.int64(20), np.int64(22), np.int64(24), np.int64(30), np.int64(31)]
  消失的神经元: 4 (在1214中存在但在1215中不存在)
    消失的神经元列表: [np.int64(5), np.int64(6), np.int64(16), np.int64(21)]
  新出现的神经元: 3 (在1215中存在但在1214中不存在)
    新出现的神经元列表: [np.int64(8), np.int64(18), np.int64(25)]
  测试recording通道数: 32

  ===== 重复实验 1/5 (使用 model_1) =====
Using old detection method: extremum_channels
Using 17 valid channels from train neuron extremum_channels
筛选eval_neuron_inf: 从16个neuron筛选到13个（移除了3个不在valid_channels中的neuron）
筛选gt_detect_array: 从22621

Noise classification: 100%|██████████| 229/229 [00:00<00:00, 233.55it/s]


Number of spikes passing noise classifier: 60684
Noise classifier准确率: 0.9696 (453034/467229)
GT spike通过noise classifier比例: 0.9262 (49587/53540)

### 5. Per-channel K-means clustering and matching

### 6. Firing rate filtering

### 7. Summary
Matching results:
  - Total clusters: 85
  - Matched clusters: 53
  - Matched neurons: 14
  - Invalid neurons (firing rate < 0.5 Hz): 0
  - Spikes matched to neurons: 56773
  - Spikes marked as noise: 3911
  - Total spikes after noise classifier: 60684

### Classification Accuracy Calculation
  Total spikes analyzed: 60684
  Overall accuracy: 0.7832 (78.32%)
  Accuracy (excluding noise): 0.9108 (91.08%)


Extracting way3 features for all spikes: 100%|██████████| 229/229 [01:23<00:00,  2.73it/s]


  重复实验 1/5 完成，结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_0/date_1215/calibration_model_1.pkl

  ===== 重复实验 2/5 (使用 model_2) =====
Using old detection method: extremum_channels
Using 17 valid channels from train neuron extremum_channels
筛选eval_neuron_inf: 从16个neuron筛选到13个（移除了3个不在valid_channels中的neuron）
筛选gt_detect_array: 从226217个spikes筛选到220933个（移除了5284个不属于valid neurons的spikes）
Stage 1: Calibration (first 60 seconds)
Loading first 300 seconds of data...
Data shape: (32, 3000000)

### 2. Threshold detection
Number of detected spikes: 516623
去重: 移除了49390个spikes（保留幅值更大的channel上的spike）
去重前: 516623个spikes, 去重后: 467233个spikes
Number of detected spikes after deduplication: 467233
GT匹配统计: 52685/53540 GT spikes被检测到 (召回率: 0.9840)

### 3. Extract waveforms

### 4. Noise classifier filtering


...

  重复实验 4/5 完成，结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_3/date_1219/calibration_model_4.pkl

  ===== 重复实验 5/5 (使用 model_5) =====
Using old detection method: extremum_channels
Using 31 valid channels from train neuron extremum_channels
筛选eval_neuron_inf: 从17个neuron筛选到16个（移除了1个不在valid_channels中的neuron）
筛选gt_detect_array: 从211316个spikes筛选到159854个（移除了51462个不属于valid neurons的spikes）
Stage 1: Calibration (first 60 seconds)
Loading first 300 seconds of data...
Data shape: (32, 3000000)

### 2. Threshold detection
Number of detected spikes: 794469
去重: 移除了165775个spikes（保留幅值更大的channel上的spike）
去重前: 794469个spikes, 去重后: 628694个spikes
Number of detected spikes after deduplication: 628694
GT匹配统计: 20663/26588 GT spikes被检测到 (召回率: 0.7772)

### 3. Extract waveforms

### 4. Noise classifier filtering


Noise classification: 100%|██████████| 307/307 [00:01<00:00, 237.86it/s]


Number of spikes passing noise classifier: 29159
Noise classifier准确率: 0.9779 (614763/628685)
GT spike通过noise classifier比例: 0.6751 (17950/26588)

### 5. Per-channel K-means clustering and matching
  Channel A-071: 12 spikes < 30, marking as noise
  Channel A-076: 8 spikes < 30, marking as noise
  Channel A-077: 21 spikes < 30, marking as noise
  Channel A-093: 17 spikes < 30, marking as noise
  Channel A-094: 24 spikes < 30, marking as noise
  Channel A-104: 5 spikes < 30, marking as noise
  Channel A-107: 3 spikes < 30, marking as noise
  Channel A-113: 10 spikes < 30, marking as noise
  Channel A-116: 6 spikes < 30, marking as noise
  Channel A-119: 4 spikes < 30, marking as noise

### 6. Firing rate filtering
  Neuron 7: firing rate 0.2333 Hz < 0.3 Hz, marked as invalid
  Neuron 20: firing rate 0.2533 Hz < 0.3 Hz, marked as invalid
  Neuron 27: firing rate 0.1533 Hz < 0.3 Hz, marked as invalid
  Marked 8 clusters and 192 spikes as noise due to low firing rate

### 7. Summary
Matching

Extracting way3 features for all spikes: 100%|██████████| 307/307 [01:27<00:00,  3.52it/s]

  重复实验 5/5 完成，结果已保存到: /media/ubuntu/sda/mouse_test/sorted/combined_mountain_sort/mouse2/clique_3/date_1219/calibration_model_5.pkl

所有测试完成！
